In [ ]:
%pip install -q torch torchvision pandas scikit-learn pillow matplotlib


In [1]:
from pathlib import Path
import random
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
DATASET_DIR = DATA_DIR / "fashion-product-images-small"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "masterCategory"
MAX_IMAGES_PER_CLASS = 400
MIN_IMAGES_PER_CLASS = 10
IMAGE_SIZE = 224
BATCH_SIZE = 32
BASELINE_EPOCHS = 3
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project:", PROJECT_DIR)
print("Dataset folder:", DATASET_DIR)
print("Device:", DEVICE)


Project: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai
Dataset folder: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai\data\fashion-product-images-small
Device: cpu


In [3]:
csv_candidates = list(DATA_DIR.rglob("styles.csv"))

if not csv_candidates:
    raise FileNotFoundError(
        "styles.csv was not found. Check that the extracted fashion-product-images-small folder is inside data/."
    )

selected_csv = None
selected_image_dir = None
df = None

for csv_path in csv_candidates:
    candidate_df = pd.read_csv(csv_path, on_bad_lines="skip")
    candidate_df.columns = candidate_df.columns.astype(str).str.strip()

    normalized_columns = {
        column.lower().replace("_", "").replace(" ", ""): column
        for column in candidate_df.columns
    }

    if "mastercategory" not in normalized_columns or "id" not in normalized_columns:
        continue

    image_candidates = [
        path for path in csv_path.parent.rglob("images")
        if path.is_dir()
    ]

    if not image_candidates:
        continue

    selected_csv = csv_path
    selected_image_dir = image_candidates[0]
    df = candidate_df
    LABEL_COLUMN = normalized_columns["mastercategory"]
    ID_COLUMN = normalized_columns["id"]
    break

if df is None:
    raise FileNotFoundError(
        "Could not find a styles.csv containing id and masterCategory together with an images folder."
    )

CSV_PATH = selected_csv
IMAGE_DIR = selected_image_dir

df.columns = df.columns.astype(str).str.strip()
df[ID_COLUMN] = df[ID_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
df["image_path"] = df[ID_COLUMN].map(lambda value: IMAGE_DIR / f"{value}.jpg")

df = df[df["image_path"].map(lambda path: path.is_file())].copy()
df = df[df[LABEL_COLUMN].notna() & (df[LABEL_COLUMN] != "")].copy()

class_counts = df[LABEL_COLUMN].value_counts()
valid_classes = class_counts[class_counts >= MIN_IMAGES_PER_CLASS].index
df = df[df[LABEL_COLUMN].isin(valid_classes)].copy()

df = (
    df.groupby(LABEL_COLUMN, group_keys=False)
      .apply(lambda group: group.sample(min(len(group), MAX_IMAGES_PER_CLASS), random_state=SEED))
      .reset_index(drop=True)
)

class_names = sorted(df[LABEL_COLUMN].unique().tolist())
class_to_idx = {name: index for index, name in enumerate(class_names)}
df["label"] = df[LABEL_COLUMN].map(class_to_idx).astype(int)

print("CSV:", CSV_PATH)
print("Images folder:", IMAGE_DIR)
print("Usable images:", len(df))
print("Number of classes:", len(class_names))
print("Classes:", class_names)
display(df[LABEL_COLUMN].value_counts())


KeyError: 'masterCategory'

In [ ]:
train_df, temporary_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"]
)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

class FashionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(row["image_path"]) as image:
            image = image.convert("RGB")
            image = self.transform(image)
        return image, int(row["label"])

train_dataset = FashionDataset(train_df, train_transform)
validation_dataset = FashionDataset(validation_df, evaluation_transform)
test_dataset = FashionDataset(test_df, evaluation_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Training images:", len(train_dataset))
print("Validation images:", len(validation_dataset))
print("Test images:", len(test_dataset))


In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, number_of_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

criterion = nn.CrossEntropyLoss()

baseline_model = CustomCNN(len(class_names)).to(DEVICE)
baseline_optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=0.001
)


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.detach().cpu().numpy())
        predicted_labels.extend(predictions.detach().cpu().numpy())

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(true_labels, predicted_labels)

    return average_loss, accuracy, true_labels, predicted_labels

baseline_history = []

for epoch in range(BASELINE_EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(
        baseline_model,
        train_loader,
        baseline_optimizer
    )

    validation_loss, validation_accuracy, _, _ = run_epoch(
        baseline_model,
        validation_loader
    )

    baseline_history.append({
        "stage": "custom_cnn_baseline",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy
    })

    print(
        f"CNN Epoch {epoch + 1}/{BASELINE_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )

baseline_test_loss, baseline_accuracy, baseline_true, baseline_pred = run_epoch(
    baseline_model,
    test_loader
)

print("Custom CNN test accuracy:", baseline_accuracy)
print(classification_report(
    baseline_true,
    baseline_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))


In [ ]:
resnet_weights = ResNet50_Weights.DEFAULT
transfer_model = models.resnet50(weights=resnet_weights)

for parameter in transfer_model.parameters():
    parameter.requires_grad = False

transfer_model.fc = nn.Linear(
    transfer_model.fc.in_features,
    len(class_names)
)

transfer_model = transfer_model.to(DEVICE)

head_optimizer = torch.optim.Adam(
    transfer_model.fc.parameters(),
    lr=0.001
)

head_history = []

for epoch in range(HEAD_EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(
        transfer_model,
        train_loader,
        head_optimizer
    )

    validation_loss, validation_accuracy, _, _ = run_epoch(
        transfer_model,
        validation_loader
    )

    head_history.append({
        "stage": "resnet50_frozen_backbone",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy
    })

    print(
        f"ResNet Head Epoch {epoch + 1}/{HEAD_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )


In [ ]:
for name, parameter in transfer_model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        parameter.requires_grad = True

fine_tune_optimizer = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, transfer_model.parameters()),
    lr=0.0001
)

fine_tune_history = []
best_validation_accuracy = -1.0
best_model_state = None

for epoch in range(FINE_TUNE_EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(
        transfer_model,
        train_loader,
        fine_tune_optimizer
    )

    validation_loss, validation_accuracy, _, _ = run_epoch(
        transfer_model,
        validation_loader
    )

    fine_tune_history.append({
        "stage": "resnet50_fine_tuning",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy
    })

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_model_state = copy.deepcopy(transfer_model.state_dict())

    print(
        f"Fine-tuning Epoch {epoch + 1}/{FINE_TUNE_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )

if best_model_state is not None:
    transfer_model.load_state_dict(best_model_state)


In [ ]:
transfer_test_loss, transfer_accuracy, transfer_true, transfer_pred = run_epoch(
    transfer_model,
    test_loader
)

print("ResNet50 test accuracy:", transfer_accuracy)
print(classification_report(
    transfer_true,
    transfer_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

comparison_df = pd.DataFrame([
    {
        "Model": "Custom CNN baseline",
        "Test Accuracy": baseline_accuracy
    },
    {
        "Model": "ResNet50 transfer learning and fine-tuning",
        "Test Accuracy": transfer_accuracy
    }
])

display(
    comparison_df.sort_values(
        "Test Accuracy",
        ascending=False
    ).reset_index(drop=True)
)

comparison_df.set_index("Model")["Test Accuracy"].plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1),
    title="Custom CNN vs ResNet50"
)

plt.ylabel("Test Accuracy")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


In [ ]:
MODEL_PATH = MODEL_DIR / "day7_resnet50_transfer_learning.pth"
BASELINE_MODEL_PATH = MODEL_DIR / "day7_custom_cnn_baseline.pth"
CLASS_NAMES_PATH = MODEL_DIR / "day7_class_names.json"
COMPARISON_PATH = REPORT_DIR / "day7_model_comparison.csv"
HISTORY_PATH = REPORT_DIR / "day7_training_history.csv"

torch.save({
    "model_state_dict": transfer_model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "architecture": "ResNet50"
}, MODEL_PATH)

torch.save({
    "model_state_dict": baseline_model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "architecture": "CustomCNN"
}, BASELINE_MODEL_PATH)

history_df = pd.DataFrame(
    baseline_history + head_history + fine_tune_history
)

comparison_df.to_csv(COMPARISON_PATH, index=False)
history_df.to_csv(HISTORY_PATH, index=False)

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=4)

print("ResNet50 model saved:", MODEL_PATH)
print("Custom CNN baseline saved:", BASELINE_MODEL_PATH)
print("Class names saved:", CLASS_NAMES_PATH)
print("Comparison saved:", COMPARISON_PATH)
print("Training history saved:", HISTORY_PATH)
